In [1]:
from google.colab import files
import os

print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Move the kaggle.json file to the hidden ~/.kaggle directory
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json


In [2]:
# 1. Download and unzip the dataset
!kaggle datasets download -d trainingdatapro/ibeta-level-1-liveness-detection-dataset-part-1
!unzip -q ibeta-level-1-liveness-detection-dataset-part-1.zip -d /content/dataset

# 2. Download the OpenCV DNN Face Detector models
!wget -q -O /content/deploy.prototxt https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
!wget -q -O /content/res10_300x300_ssd_iter_140000_fp16.caffemodel https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20180205_fp16/res10_300x300_ssd_iter_140000_fp16.caffemodel

# 3. Download the MediaPipe Face Landmarker task file
!wget -q -O /content/face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

# 4. Create the project directory
!mkdir /content/project

Dataset URL: https://www.kaggle.com/datasets/trainingdatapro/ibeta-level-1-liveness-detection-dataset-part-1
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 1.83G/1.83G [00:20<00:00, 95.2MB/s]



In [7]:
import cv2
import os
import glob
import sys

# 1. Change the notebook's working directory so "../" correctly points to "/content/"
os.chdir('/content/project')

# 2. Append path and import
sys.path.append('/content/project')
from utils import detect_largest_face, crop_face

VIDEO_DIR = "/content/dataset"
OUTPUT_DIR = "/content/processed_dataset"

os.makedirs(os.path.join(OUTPUT_DIR, "real"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "fake"), exist_ok=True)

# Grab all mp4 files
video_paths = glob.glob(os.path.join(VIDEO_DIR, "**", "*.mp4"), recursive=True)
print(f"Found {len(video_paths)} videos to process...")

FRAME_SKIP = 30
MAX_FRAMES_PER_VIDEO = 5

for i, video_path in enumerate(video_paths):
    # Determine label based on folder structure
    if "Real" in video_path or "real" in video_path.lower():
        label = "real"
    else:
        label = "fake"

    cap = cv2.VideoCapture(video_path)
    count = 0
    saved_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if count % FRAME_SKIP == 0:
            box = detect_largest_face(frame)
            if box is not None:
                face = crop_face(frame, box)
                if face is not None:
                    # Save the cropped face
                    filename = f"vid{i}_f{count}.jpg"
                    save_path = os.path.join(OUTPUT_DIR, label, filename)
                    cv2.imwrite(save_path, face)
                    saved_count += 1

        count += 1
        if saved_count >= MAX_FRAMES_PER_VIDEO:
            break

    cap.release()

print("Preprocessing complete! Dataset is ready.")

Found 86 videos to process...
Preprocessing complete! Dataset is ready.


In [19]:
%cd /content/project
!python train.py --dataset /content/processed_dataset --model /content/liveness_final.model --le /content/le_final.pickle --plot /content/plot_final.png

/content
[INFO] loading images...
[INFO] compiling model...
[INFO] training network for 100 epochs...
Epoch 1/100 - loss: 0.7305 - accuracy: 0.5656 - val_loss: 1.4962 - val_accuracy: 0.1393
Epoch 2/100 - loss: 0.6983 - accuracy: 0.5902 - val_loss: 0.9410 - val_accuracy: 0.2459
Epoch 3/100 - loss: 0.6854 - accuracy: 0.5820 - val_loss: 0.9181 - val_accuracy: 0.3033
Epoch 4/100 - loss: 0.6035 - accuracy: 0.6585 - val_loss: 0.9422 - val_accuracy: 0.2951
Epoch 5/100 - loss: 0.6393 - accuracy: 0.6366 - val_loss: 0.9378 - val_accuracy: 0.3197
Epoch 6/100 - loss: 0.6264 - accuracy: 0.6721 - val_loss: 1.1011 - val_accuracy: 0.2623
Epoch 7/100 - loss: 0.6037 - accuracy: 0.6831 - val_loss: 0.9263 - val_accuracy: 0.4344
Epoch 8/100 - loss: 0.5742 - accuracy: 0.7077 - val_loss: 0.8261 - val_accuracy: 0.5246
Epoch 9/100 - loss: 0.5258 - accuracy: 0.7404 - val_loss: 0.7157 - val_accuracy: 0.7131
Epoch 10/100 - loss: 0.5234 - accuracy: 0.7186 - val_loss: 0.7061 - val_accuracy: 0.8033
Epoch 11/100 - lo

In [18]:
!rm /content/processed_dataset/real/vid*